In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install pyserial scipy

In [ ]:
import time, json, threading, queue
from pathlib import Path
import numpy as np
from scipy.stats import kurtosis, skew
import serial

WARMUP_DURATION   = 30 * 60
COLLECT_DURATION  = 90 * 60
SAMPLING_RATE     = 10_000
WINDOW_SIZE       = 1024
WINDOW_OVERLAP    = 512
WINDOW_STEP       = WINDOW_SIZE - WINDOW_OVERLAP
OUTPUT_DIR        = Path(".")
FEATURES_FILE     = OUTPUT_DIR / "features.npy"
META_FILE         = OUTPUT_DIR / "collection_meta.json"

In [ ]:
FEATURE_NAMES = [
    "mean", "std", "rms", "max", "min",
    "peak_to_peak", "skewness", "kurtosis",
    "crest_factor", "shape_factor", "impulse_factor",
    "freq_dominant", "spectral_energy", "spectral_entropy",
]

def extract_features(window: np.ndarray, fs: int = SAMPLING_RATE) -> np.ndarray:
    n = len(window)
    eps = 1e-10

    mean_val    = np.mean(window)
    std_val     = np.std(window)
    rms_val     = np.sqrt(np.mean(window ** 2))
    max_val     = np.max(np.abs(window))
    min_val     = np.min(np.abs(window))
    p2p_val     = np.ptp(window)
    skew_val    = float(skew(window))
    kurt_val    = float(kurtosis(window))
    crest_val   = max_val / (rms_val + eps)
    mean_abs    = np.mean(np.abs(window))
    shape_val   = rms_val  / (mean_abs + eps)
    impulse_val = max_val  / (mean_abs + eps)

    fft_mag     = np.abs(np.fft.rfft(window))
    freqs       = np.fft.rfftfreq(n, d=1.0 / fs)
    freq_dom    = freqs[np.argmax(fft_mag)]
    spec_energy = np.sum(fft_mag ** 2)
    psd_norm    = fft_mag ** 2 / (spec_energy + eps)
    spec_entropy = -np.sum(psd_norm * np.log2(psd_norm + eps))

    return np.array([
        mean_val, std_val, rms_val, max_val, min_val,
        p2p_val, skew_val, kurt_val, crest_val, shape_val,
        impulse_val, freq_dom, spec_energy, spec_entropy,
    ], dtype=np.float32)

In [ ]:
def reader_thread(port, baud, data_queue, stop_event):
    try:
        ser = serial.Serial(port, baud, timeout=1.0)
        print(f"[Capteur] Connecté sur {port} @ {baud} baud")
        while not stop_event.is_set():
            line = ser.readline().decode("utf-8", errors="ignore").strip()
            if not line:
                continue
            try:
                value = float(line.split(",")[0])
                data_queue.put(value)
            except ValueError:
                pass
        ser.close()
    except serial.SerialException as e:
        print(f"[Capteur] Erreur série : {e}")
        stop_event.set()

def simulate_reader(data_queue, stop_event, fs=SAMPLING_RATE):
    print("[Simulation] Signal synthétique : 120 Hz + bruit")
    t = 0
    dt = 1.0 / fs
    while not stop_event.is_set():
        value = 0.5 * np.sin(2 * np.pi * 120 * t) + 0.05 * np.random.randn()
        data_queue.put(float(value))
        t += dt
        if int(t * fs) % 100 == 0:
            time.sleep(100 / fs)

In [ ]:
def collect(port=None, baud=115200, simulate=False):
    data_queue    = queue.Queue(maxsize=100_000)
    stop_event    = threading.Event()
    buffer        = []
    features_list = []

    if simulate:
        t = threading.Thread(target=simulate_reader, args=(data_queue, stop_event), daemon=True)
    else:
        t = threading.Thread(target=reader_thread,   args=(port, baud, data_queue, stop_event), daemon=True)
    t.start()

    start_time = time.time()

    # ── Phase rodage ──────────────────────────────────────────────────────────
    print(f"[Phase 1] Rodage ({WARMUP_DURATION//60} min) — données ignorées...")
    while time.time() - start_time < WARMUP_DURATION:
        remaining = WARMUP_DURATION - (time.time() - start_time)
        print(f"\r  {int(remaining//60):02d}:{int(remaining%60):02d} restantes   ", end="", flush=True)
        while not data_queue.empty():
            data_queue.get_nowait()
        time.sleep(5)
    print("\n[Phase 1] Rodage terminé ✓")

    # ── Phase collecte ────────────────────────────────────────────────────────
    collect_start = time.time()
    print(f"[Phase 2] Collecte ({COLLECT_DURATION//60} min) — extraction des features...")

    while time.time() - collect_start < COLLECT_DURATION:
        while not data_queue.empty() and len(buffer) < WINDOW_SIZE * 4:
            buffer.append(data_queue.get_nowait())

        while len(buffer) >= WINDOW_SIZE:
            window = np.array(buffer[:WINDOW_SIZE], dtype=np.float32)
            buffer = buffer[WINDOW_STEP:]
            features_list.append(extract_features(window))

        remaining = COLLECT_DURATION - (time.time() - collect_start)
        print(f"\r  {int(remaining//60):02d}:{int(remaining%60):02d} restantes "
              f"| {len(features_list)} fenêtres   ", end="", flush=True)
        time.sleep(0.1)

    stop_event.set()
    print(f"\n[Phase 2] Terminé ✓ — {len(features_list)} fenêtres extraites")

    if not features_list:
        print("[ERREUR] Aucune feature extraite.")
        return

    # ── Sauvegarde ────────────────────────────────────────────────────────────
    features_array = np.stack(features_list, axis=0)
    np.save(FEATURES_FILE, features_array)
    print(f"[Sauvegarde] {FEATURES_FILE}  shape={features_array.shape}")

    meta = {
        "n_windows"       : int(features_array.shape[0]),
        "n_features"      : int(features_array.shape[1]),
        "feature_names"   : FEATURE_NAMES,
        "window_size"     : WINDOW_SIZE,
        "window_step"     : WINDOW_STEP,
        "sampling_rate_hz": SAMPLING_RATE,
        "collected_at"    : time.strftime("%Y-%m-%dT%H:%M:%S"),
        "features_mean"   : features_array.mean(axis=0).tolist(),
        "features_std"    : features_array.std(axis=0).tolist(),
    }
    with open(META_FILE, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"[Sauvegarde] {META_FILE}")

    print("\n── Aperçu des moyennes ──")
    for name, val in zip(FEATURE_NAMES, features_array.mean(axis=0)):
        print(f"  {name:<20} : {val:.4f}")

# ── Lancement ─────────────────────────────────────────────────────────────────
collect(simulate=True)       # ← test sans capteur
# collect(port="/dev/ttyUSB0", baud=115200)   # ← avec capteur réel

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow import keras

FEATURES_FILE  = Path("features.npy")
META_FILE      = Path("collection_meta.json")
MODEL_FILE     = Path("autoencoder.h5")
SCALER_FILE    = Path("scaler.npy")
THRESHOLD_FILE = Path("threshold.json")

K_SIGMA = 3.0   # seuil = mean + K_SIGMA × std de l'erreur de reconstruction

# Chargement
features = np.load(FEATURES_FILE)
with open(META_FILE) as f:
    meta = json.load(f)

print(f"Features chargées : shape={features.shape}")
print(f"Fenêtres : {meta['n_windows']}  |  Features : {meta['n_features']}")

In [ ]:
scaler = RobustScaler()
features_scaled = scaler.fit_transform(features)

# Sauvegarde des paramètres du scaler (center_ et scale_)
np.save(SCALER_FILE, np.array([scaler.center_, scaler.scale_]))
print(f"Scaler sauvegardé → {SCALER_FILE}")
print(f"  center_ : {scaler.center_[:4].round(4)} ...")
print(f"  scale_  : {scaler.scale_[:4].round(4)} ...")

In [ ]:
N_FEATURES = features_scaled.shape[1]   # 14

inputs   = keras.Input(shape=(N_FEATURES,))
encoded  = keras.layers.Dense(8,  activation="relu")(inputs)
encoded  = keras.layers.Dense(4,  activation="relu")(encoded)   # bottleneck
decoded  = keras.layers.Dense(8,  activation="relu")(encoded)
outputs  = keras.layers.Dense(N_FEATURES, activation="linear")(decoded)

autoencoder = keras.Model(inputs, outputs, name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

In [ ]:
history = autoencoder.fit(
    features_scaled, features_scaled,   # X = Y (reconstruction)
    epochs          = 50,
    batch_size      = 64,
    validation_split= 0.1,
    shuffle         = True,
    callbacks       = [
        keras.callbacks.EarlyStopping(
            monitor  = "val_loss",
            patience = 5,
            restore_best_weights = True
        )
    ],
    verbose = 1
)

autoencoder.save(MODEL_FILE)
print(f"\nModèle sauvegardé → {MODEL_FILE}")

In [ ]:
# Erreur de reconstruction sur toutes les données d'entraînement
reconstructions = autoencoder.predict(features_scaled, verbose=0)
errors = np.mean((features_scaled - reconstructions) ** 2, axis=1)  # MSE par fenêtre

mean_err = float(np.mean(errors))
std_err  = float(np.std(errors))
threshold = mean_err + K_SIGMA * std_err

print(f"Erreur moyenne  : {mean_err:.6f}")
print(f"Erreur std      : {std_err:.6f}")
print(f"Seuil (mean + {K_SIGMA}σ) : {threshold:.6f}")

threshold_data = {
    "mean_error" : mean_err,
    "std_error"  : std_err,
    "k_sigma"    : K_SIGMA,
    "threshold"  : threshold,
    "scaler_center" : scaler.center_.tolist(),
    "scaler_scale"  : scaler.scale_.tolist(),
}
with open(THRESHOLD_FILE, "w") as f:
    json.dump(threshold_data, f, indent=2)
print(f"Seuil sauvegardé → {THRESHOLD_FILE}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(errors, bins=80, color="steelblue", edgecolor="white", alpha=0.8)
plt.axvline(threshold, color="red",    linestyle="--", linewidth=2, label=f"Seuil ({K_SIGMA}σ) = {threshold:.4f}")
plt.axvline(mean_err,  color="orange", linestyle="--", linewidth=1, label=f"Moyenne = {mean_err:.4f}")
plt.xlabel("Erreur de reconstruction (MSE)")
plt.ylabel("Nombre de fenêtres")
plt.title("Distribution des erreurs — données normales d'entraînement")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import hls4ml
print(hls4ml.__version__)

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import json
import numpy as np
from pathlib import Path
import tensorflow as tf
import hls4ml

MODEL_FILE     = Path("autoencoder.h5")
FEATURES_FILE  = Path("features.npy")
HLS_OUTPUT_DIR = Path("hls4ml_output")

# Reuse factors à explorer (plus RF est grand → moins de ressources, plus de latence)
REUSE_FACTORS = [1, 2, 4, 8]

model = tf.keras.models.load_model(MODEL_FILE)
model.summary()

In [ ]:
def convert_model(model, reuse_factor: int, output_dir: Path):
    config = hls4ml.utils.config_from_keras_model(
        model,
        granularity="name",
        default_precision="ap_fixed<16,6>",
        default_reuse_factor=reuse_factor
    )

    print(f"\n── Config RF={reuse_factor} ──")
    for layer_name, layer_cfg in config["LayerName"].items():
        print(f"  {layer_name}: {layer_cfg}")

    hls_model = hls4ml.converters.convert_from_keras_model(
        model,
        hls_config = config,
        output_dir = str(output_dir / f"rf{reuse_factor}"),
        backend    = "VivadoAccelerator",
        board      = "pynq-z2",
        interface  = "axi_stream",
    )

    hls_model.compile()
    return hls_model

In [ ]:
hls_models = {}

for rf in REUSE_FACTORS:
    print(f"\n{'='*50}")
    print(f"  Conversion RF={rf}")
    print(f"{'='*50}")
    try:
        hls_m = convert_model(model, rf, HLS_OUTPUT_DIR)
        hls_models[rf] = hls_m
        print(f"[RF={rf}] ✓ Conversion OK")
    except Exception as e:
        print(f"[RF={rf}] ✗ Erreur : {e}")

In [ ]:
features = np.load(FEATURES_FILE)

# Normalise avec le scaler sauvegardé
scaler_params = np.load("scaler.npy")
center = scaler_params[0]
scale  = scaler_params[1]
features_scaled = (features - center) / scale

# Prend un échantillon de 1000 fenêtres pour le test
X_test = features_scaled[:1000].astype(np.float32)

print(f"{'RF':<6} {'MSE keras':>12} {'MSE hls4ml':>12} {'Diff %':>10}")
print("-" * 44)

keras_pred = model.predict(X_test, verbose=0)
keras_mse  = float(np.mean((X_test - keras_pred) ** 2))

for rf, hls_m in hls_models.items():
    hls_pred = hls_m.predict(X_test)
    hls_mse  = float(np.mean((X_test - hls_pred) ** 2))
    diff_pct = abs(hls_mse - keras_mse) / (keras_mse + 1e-10) * 100
    print(f"{rf:<6} {keras_mse:>12.6f} {hls_mse:>12.6f} {diff_pct:>9.2f}%")

In [ ]:
BEST_RF = 4

print(f"Lancement de la synthèse Vivado pour RF={BEST_RF}...")
print("⚠️  Cette étape prend 20-40 minutes, ne ferme pas le notebook.\n")

best_model = hls_models[BEST_RF]
report = best_model.build(
    csim    = False,
    synth   = True,
    export  = True,
)

print("\n── Rapport de synthèse ──")
hls4ml.report.read_vivado_report(
    str(HLS_OUTPUT_DIR / f"rf{BEST_RF}")
)

In [ ]:
hls4ml.report.read_vivado_report(str(HLS_OUTPUT_DIR / f"rf{BEST_RF}"))

In [ ]:
import json
import hls4ml

# Trouve le fichier supported_board.json
import os
hls4ml_path = os.path.dirname(hls4ml.__file__)
board_file  = os.path.join(hls4ml_path, "backends", "vivado_accelerator", "supported_board.json")

with open(board_file) as f:
    boards = json.load(f)

# Affiche ce qui est disponible pour pynq-z2
print(json.dumps(boards.get("pynq-z2", {}), indent=2))

In [ ]:
import subprocess
result = subprocess.run(
    ["find", hls4ml_path, "-name", "*.json", "-path", "*board*"],
    capture_output=True, text=True
)
print(result.stdout)

# Et aussi cherche tous les json qui parlent de pynq
result2 = subprocess.run(
    ["grep", "-r", "pynq", hls4ml_path, "--include=*.json", "-l"],
    capture_output=True, text=True
)
print(result2.stdout)

In [ ]:
board_file = os.path.join(hls4ml_path, "backends", "vivado_accelerator", "supported_boards.json")

with open(board_file) as f:
    boards = json.load(f)

print(json.dumps(boards.get("pynq-z2", {}), indent=2))

In [ ]:
import subprocess
result = subprocess.run(
    ["find", 
     "/home/moi/miniconda3/envs/hls4ml-tutorial/lib/python3.10/site-packages/hls4ml/templates/vivado_accelerator/pynq-z2",
     "-type", "f"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
best_model.build(
    csim    = False,
    synth   = False,
    export  = False,
    bitfile = True,
)
print("Bitstream généré ✓")